# Groundwater exploration — raster & vector inventory

Scan a folder of GIS layers, create lighter optimized copies, and summarize/visualize what is inside it:

1. **Optimize** — convert `.tif` → Cloud Optimized GeoTIFF (COG) and `.shp` → GeoParquet
2. **Raster (`.tif`) inventory** — CRS and pixel resolution per file
3. **Vector inventory** — CRS, number of rows (features), and list of columns per file
4. **Raster plots** — quick-look preview of every raster
5. **Vector dynamic maps** — interactive Folium map per vector layer

Paste the folder path in the **Define input folder** cell below, then run all cells.


### Import packages


In [1]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from rio_cogeo.cogeo import cog_translate
from rio_cogeo.profiles import cog_profiles


### Define input folder

Paste the folder path containing the `.tif` and `.shp` files below.


In [2]:
# TODO: paste the folder path containing the .tif and .shp files
data_dir = Path(r"P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian")

if not data_dir.is_dir():
    raise FileNotFoundError(f"Folder not found: {data_dir}")

print("Data folder:", data_dir)


Data folder: P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian


### Optimize large files (COG + GeoParquet)

The source `.tif`/`.shp` files can be very large, which makes them slow to read and plot. This first step creates lighter copies before any summarizing/plotting happens:

- Each `.tif` is rewritten as a **Cloud Optimized GeoTIFF (COG)** — internally tiled, DEFLATE-compressed, with overviews (pyramids) so it can be read/rendered at reduced resolution without loading the full array. COGs keep the `.tif` extension: COG is a specific internal layout of a regular GeoTIFF, not a new file format.
- Each `.shp` (plus its `.dbf`/`.shx`/`.prj`/... sidecar files) is rewritten as a single **GeoParquet** file (`.parquet`), which is typically much smaller and faster to read than a shapefile.

Outputs are written to `optimized_dir` below, keeping the original file name (only the vector extension changes, from `.shp` to `.parquet`).


In [3]:
optimized_dir = Path(
    r"P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\optimized"
)
optimized_dir.mkdir(parents=True, exist_ok=True)

print("Optimized output folder:", optimized_dir)


Optimized output folder: P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\optimized


#### Rasters → Cloud Optimized GeoTIFF (COG)


In [4]:
tif_paths_to_optimize = sorted(data_dir.rglob("*.tif"))
print(f"Found {len(tif_paths_to_optimize)} .tif file(s) to optimize")

cog_profile = cog_profiles.get("deflate")

cog_paths = []
for tif_path in tif_paths_to_optimize:
    out_path = optimized_dir / tif_path.name  # COG keeps the .tif extension
    cog_translate(
        str(tif_path),
        str(out_path),
        cog_profile,
        overview_level=5,
        overview_resampling="average",
        quiet=True,
    )
    cog_paths.append(out_path)
    before_mb = tif_path.stat().st_size / 1e6
    after_mb = out_path.stat().st_size / 1e6
    print(f"{tif_path.name}: {before_mb:.1f} MB -> {after_mb:.1f} MB (COG)")

print(f"\nWrote {len(cog_paths)} COG(s) to {optimized_dir}")


Found 21 .tif file(s) to optimize
clay_5km_0-5cm.tif: 14.8 MB -> 15.1 MB (COG)
clay_5km_100-200cm.tif: 15.3 MB -> 15.5 MB (COG)
clay_5km_15-30cm.tif: 15.3 MB -> 15.5 MB (COG)
clay_5km_30-60cm.tif: 15.6 MB -> 15.8 MB (COG)
clay_5km_5-15cm.tif: 15.1 MB -> 15.4 MB (COG)
clay_5km_60-100cm.tif: 15.0 MB -> 15.3 MB (COG)
gebco_2025_sub_ice_n0.0_s-90.0_w-90.0_e0.0.tif: 933.3 MB -> 597.2 MB (COG)
gebco_2025_sub_ice_n0.0_s-90.0_w0.0_e90.0.tif: 933.3 MB -> 604.4 MB (COG)
gebco_2025_sub_ice_n90.0_s0.0_w-90.0_e0.0.tif: 933.3 MB -> 620.0 MB (COG)
gebco_2025_sub_ice_n90.0_s0.0_w0.0_e90.0.tif: 933.3 MB -> 531.9 MB (COG)


FileNotFoundError: [WinError 53] The network path was not found: 'P:\\11209117-041-ipdc-egypt-swi\\data\\IntDeltaPlatform_QGA\\Data_Vivian\\optimized\\tmpjk55vscv.tif'

#### Vectors → GeoParquet


In [ ]:
shp_paths_to_optimize = sorted(data_dir.rglob("*.shp"))
print(f"Found {len(shp_paths_to_optimize)} .shp file(s) to optimize")

parquet_paths = []
for shp_path in shp_paths_to_optimize:
    gdf = gpd.read_file(shp_path)
    out_path = optimized_dir / f"{shp_path.stem}.parquet"
    gdf.to_parquet(out_path)
    parquet_paths.append(out_path)

    # A shapefile is really a set of sidecar files (.shp/.dbf/.shx/.prj/...);
    # sum them all for a fair "before" size comparison.
    before_mb = sum(
        f.stat().st_size for f in shp_path.parent.glob(f"{shp_path.stem}.*")
    ) / 1e6
    after_mb = out_path.stat().st_size / 1e6
    print(f"{shp_path.name}: {before_mb:.1f} MB -> {after_mb:.1f} MB (GeoParquet)")

print(f"\nWrote {len(parquet_paths)} GeoParquet file(s) to {optimized_dir}")


### Choose which copies to explore below

The summaries/plots/maps further down read from `raster_read_dir` / `vector_read_dir`. By default these point at the optimized COG/GeoParquet copies (fast, recommended for large datasets). Set `USE_OPTIMIZED = False` to read the original `.tif`/`.shp` files instead.


In [ ]:
USE_OPTIMIZED = True

raster_read_dir = optimized_dir if USE_OPTIMIZED else data_dir
vector_read_dir = optimized_dir if USE_OPTIMIZED else data_dir
vector_glob_pattern = "*.parquet" if USE_OPTIMIZED else "*.shp"

print("Reading rasters from:", raster_read_dir)
print("Reading vectors from:", vector_read_dir, f"(pattern: {vector_glob_pattern})")


### 1) Raster (`.tif`) summary

For every `.tif` file found (recursively) under `raster_read_dir`, report the CRS and the pixel resolution (x, y), plus a few extra raster properties (size, band count, dtype).


In [ ]:
tif_paths = sorted(raster_read_dir.rglob("*.tif"))
print(f"Found {len(tif_paths)} .tif file(s)")

raster_records = []
for tif_path in tif_paths:
    with rasterio.open(tif_path) as src:
        res_x, res_y = src.res
        raster_records.append(
            {
                "file": tif_path.relative_to(raster_read_dir),
                "crs": src.crs.to_string() if src.crs else None,
                "resolution_x": res_x,
                "resolution_y": res_y,
                "width": src.width,
                "height": src.height,
                "band_count": src.count,
                "dtype": src.dtypes[0] if src.dtypes else None,
            }
        )

raster_summary = pd.DataFrame(raster_records)
raster_summary


### 2) Vector (`.shp`/`.parquet`) summary

For every vector file found (recursively) under `vector_read_dir`, report the CRS, the number of rows (features), and the list of columns.


In [ ]:
shp_paths = sorted(vector_read_dir.rglob(vector_glob_pattern))
print(f"Found {len(shp_paths)} vector file(s) ({vector_glob_pattern})")

vector_records = []
for shp_path in shp_paths:
    gdf = (
        gpd.read_parquet(shp_path)
        if shp_path.suffix == ".parquet"
        else gpd.read_file(shp_path)
    )
    vector_records.append(
        {
            "file": shp_path.relative_to(vector_read_dir),
            "crs": gdf.crs.to_string() if gdf.crs else None,
            "n_rows": len(gdf),
            "columns": list(gdf.columns),
        }
    )

vector_summary = pd.DataFrame(vector_records)
vector_summary


### 3) Raster (`.tif`) plots

Plot every raster in a grid so they can be visually inspected (extent, values,
obvious artefacts, no-data areas, etc.).


In [ ]:
max_preview_size = 1024  # longest side in pixels; uses COG overviews when available
n_cols = 3
n_rows = -(-len(tif_paths) // n_cols) if tif_paths else 0

fig, axes = plt.subplots(
    n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows), squeeze=False
)

for ax, tif_path in zip(axes.flat, tif_paths):
    with rasterio.open(tif_path) as src:
        scale = min(1, max_preview_size / max(src.width, src.height))
        out_shape = (max(1, int(src.height * scale)), max(1, int(src.width * scale)))
        band = src.read(
            1, out_shape=out_shape, masked=True, resampling=Resampling.average
        )
        img = ax.imshow(band, cmap="viridis")
        ax.set_title(tif_path.name, fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
        fig.colorbar(img, ax=ax, fraction=0.046, pad=0.04)

# Hide any unused subplot axes (grid is not always fully filled)
for ax in axes.flat[len(tif_paths) :]:
    ax.axis("off")

fig.tight_layout()
plt.show()


### 4) Vector (`.shp`) dynamic maps

Render every shapefile as an interactive (Folium-based) map via
`GeoDataFrame.explore()`, so individual features can be inspected/clicked.
Each map is stored in `vector_maps`, keyed by filename.


In [ ]:
vector_maps = {}
for shp_path in shp_paths:
    gdf = (
        gpd.read_parquet(shp_path)
        if shp_path.suffix == ".parquet"
        else gpd.read_file(shp_path)
    )
    vector_maps[shp_path.name] = gdf.explore(
        tooltip=True,
        popup=True,
        style_kwds={"fillOpacity": 0.4},
    )

# Display one map at a time by picking its filename, e.g.:
first_name = next(iter(vector_maps))
print("Showing:", first_name)
vector_maps[first_name]
